# Running remote commands with `htp.ssh`

The `htp.ssh` module wraps a local `ssh` executable so you can run one or more commands on a remote host from Python. It returns the combined stdout and stderr as a string and raises `SSHError` if the remote command exits with a non-zero status.

The code cells below are illustrative. They are not executed when the book builds, because they need a live SSH target.

## Importing

The two names you need are `ssh` and `SSHError`. Both are re-exported from the top level of the package.

In [ ]:
from htp import ssh, SSHError

## A single command

Pass the command as a string and the SSH target as `user@host` (or just `host` if your SSH config already knows the user). The call blocks until the remote process finishes, then returns a `(status, output)` pair, where `output` is the combined stdout and stderr of the remote process.

In [ ]:
status, output = ssh("uname -a", "jkitchin@beowulf.cheme.cmu.edu")
print(output)

## Several commands in one session

If you pass an iterable of strings, `htp.ssh` joins them with `;` and sends them as a single remote command. All of the commands therefore share the same working directory and environment on the remote host.

In [ ]:
status, output = ssh(
    ["cd work", "ls", "du -sh ."],
    "jkitchin@beowulf.cheme.cmu.edu",
)
print(output)

## Handling failures

A non-zero exit status raises `SSHError` with the failed argv and captured output attached to the message. Catch it when the remote command is expected to fail sometimes, for example when probing for the existence of a file.

In [ ]:
try:
    ssh("ls /does/not/exist", "jkitchin@beowulf.cheme.cmu.edu")
except SSHError as err:
    print("remote check failed:")
    print(err)

## Customizing the ssh command

The `ssh_cmd` argument is the argv prefix used to launch ssh. The default is `("ssh", "-x")`, which disables X11 forwarding. Override it to pass a different identity file, port, connection timeout, or any other ssh option.

Because `htp.ssh` builds an argv list and calls `subprocess.run` without a shell, each element of `ssh_cmd` is treated as a separate argv entry. Do not pre-quote values or glue flags and values together with spaces inside one string.

In [ ]:
ssh(
    "hostname",
    "jkitchin@beowulf.cheme.cmu.edu",
    ssh_cmd=("ssh", "-x", "-i", "~/.ssh/htp_id", "-o", "ConnectTimeout=5"),
)

## Notes and caveats

The local side does not interpolate a shell, so local shell metacharacters in the host or command arguments are safe. The remote side, however, still executes the joined command string through the login shell on the remote host. Quoting rules on the remote side therefore apply as usual: if you need a literal semicolon or quote in a command, escape it for the remote shell, not the local one.

Authentication is delegated entirely to ssh. Use ssh keys, `ssh-agent`, or your `~/.ssh/config` to configure it. `htp.ssh` does not know about passwords.